# 01 · Connect to the database and validate data quality

**Assignment 5, step 1.** Connects to the SQLite database built in Assignment 4
(`uv run python scripts/run_pipeline.py`) and checks that the data can be trusted before any analysis.

All logic lives in [`src/analysis/exploratory.py`](../src/analysis/exploratory.py); this notebook only calls it.
If the real database hasn't been built, the notebook falls back to the synthetic sample and says so.

In [1]:
from functools import partial

import pandas as pd
from IPython.display import display

from src.analysis import charts as ch
from src.analysis import exploratory as ex
from src.analysis import statistical as st

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")
con, db_path = ex.open_database()   # real database, or the synthetic sample if it isn't built
IS_REAL = db_path == ex.REAL_DB
print(f"Connected to {db_path.name} ({'real data' if IS_REAL else 'SYNTHETIC SAMPLE: numbers are fake'})")

Connected to rare_earth.db (real data)


## 1. Data-quality checks

`check_database()` runs each check against the database itself:
- **SQLite level:** file integrity, foreign keys on, no orphan rows.
- **Tables:** none of the five stored tables is empty.
- **Views:** every analysis view has one row per key and passes the same Pandera rules the pipeline uses.
- **Cross-check:** the merged view keeps every trade row except re-imports.

In [2]:
checks = ex.check_database(con)
print(f"{checks['passed'].sum()} of {len(checks)} checks passed")
checks

19 of 19 checks passed


,check,passed,detail
0,Foreign keys enforced on this connection,True,PRAGMA foreign_keys = 1
1,SQLite file integrity,True,ok
2,No orphan rows,True,0 rows without a matching parent
3,Table countries is not empty,True,73 rows
4,Table products is not empty,True,1 rows
5,Table energy_sources is not empty,True,17 rows
6,Table trade_flows is not empty,True,227 rows
7,Table electricity_generation is not empty,True,"6,729 rows"
8,View electricity_generation_detail: one row pe...,True,"6,729 rows, 0 duplicate keys on (country, year..."
9,View electricity_generation_detail: passes val...,True,all Pandera checks pass


## 2. Row counts

In [3]:
ex.table_row_counts(con).to_frame()

,rows
countries,73
products,1
energy_sources,17
trade_flows,227
electricity_generation,6729


## 3. Coverage: which years does each trade direction have?

This is the most important limitation for the analysis. `trade_flow` is from China's point of view:
**Export** = a country buys from China, **Import** = China buys from the country.

In [4]:
ex.coverage_by_year(con)

,Export,Import
year,,
2020,0,25
2021,0,26
2022,0,25
2023,60,0
2024,60,0
2025,0,31


In the real data, no year has both directions. Export rows exist only for 2023–2024, and Import rows
only for 2020–2022 and 2025. So:
- **Imports from China** can only be studied for 2023–2024, and their **growth** only for 2024.
- Comparing Export and Import rows also compares **different years**, so direction and year are confounded.

Re-downloading `data1.xlsx` with both directions for every year would remove this limitation.

## 4. Missing values in the country-year panel

In [5]:
panel = ex.load_country_panel(con)
cols = ["imports_from_china_kg", "import_growth_pct", "wind_generation_twh",
        "wind_generation_change_twh", "wind_capacity_gw", "total_generation_twh"]
ex.missing_by_year(panel, cols)

,imports_from_china_kg,import_growth_pct,wind_generation_twh,wind_generation_change_twh,wind_capacity_gw,total_generation_twh
year,,,,,,
2020,100.00,100.00,5.60,6.90,5.60,4.20
2021,100.00,100.00,5.60,5.60,5.60,4.20
2022,100.00,100.00,5.60,5.60,5.60,4.20
2023,16.70,100.00,6.90,6.90,6.90,5.60
2024,16.70,27.80,6.90,6.90,8.30,5.60
2025,100.00,100.00,18.10,18.10,26.40,16.70


**Reading the table**
- **Imports:** missing in every year without Export rows (see section 3); recorded for 2023–2024 only.
- **Growth:** also needs the previous year, so it exists for 2024 only.
- **Wind:** missing for the few partners with no Ember data (e.g. Peru, Venezuela, "Other Asia, nes").

## Conclusion

The database passes every integrity and validation check, so the data is internally consistent. The
limits are about **coverage**, not quality: one trade direction per year, and imports only for
2023–2024. Notebook 02 works within these limits and states them next to each finding.